In [ ]:
import sys
import os

from easymgtd.auto import AutoDetector, AutoExperiment
from easymgtd.loading.dataloader import load

from dotenv import load_dotenv

load_dotenv()

# Load model paths from env
roberta_path = os.environ.get("ROBERTA_BASE", "FacebookAI/roberta-base")
distilbert_path = os.environ.get(
    "DISTILBERT_BASE", "distilbert/distilbert-base-uncased"
)
gpt2 = os.environ.get("GPT2_MEDIUM", "openai-community/gpt2-medium")

# fast-detectGPT
gpt_neo = os.environ.get("GPT_NEO", "EleutherAI/gpt-neo-2.7B")
gpt_j = os.environ.get("GPT_J", "EleutherAI/gpt-j-6b")

# binoculars
falcon_7b = os.environ.get("FALCON_7B", "tiiuae/falcon-7b")
falcon_7b_instruct = os.environ.get("FALCON_7B_INSTRUCT", "tiiuae/falcon-7b-instruct")

llama2_chat = os.environ.get("LLAMA2_CHAT", "meta-llama/Llama-2-7b-chat-hf")
radar_model = os.environ.get("RADAR_MODEL", "TrustSafeAI/RADAR-Vicuna-7B")
chatgpt_roberta = os.environ.get(
    "CHATGPT_ROBERTA", "Hello-SimpleAI/chatgpt-detector-roberta"
)

config_path = "/data/liyang/MGTD-Baselines/EasyMGTD/run/config.json"
config = {}
if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)
else:
    print(f"Warning: Config file {config_path} not found. Proceeding with defaults.")

In [ ]:
# 1. 构建验证数据集 (Mock Data)
mock_data = {
    "train": {
        "text": [
            "Natural language processing is a subfield of linguistics and artificial intelligence.",
            "The quick brown fox jumps over the lazy dog.",
            "As an AI language model, I don't have personal opinions.",
            "This text was absolutely generated by a machine model.",
            "A traditional method for creating deep learning architectures.",
            "Let's figure out how this machine learning model works.",
        ],
        "label": [0, 0, 1, 1, 0, 1],
    },
    "test": {
        "text": [
            "Machine learning models are trained on large datasets.",
            "I am designed to generate human-like text responses.",
            "This is a human written text for testing purposes.",
            "AI generated paragraphs are everywhere nowadays.",
        ],
        "label": [0, 1, 0, 1],
    },
}

In [ ]:
# 2. 全局实验配置 (Config)
class TestConfig:
    # 基础模型
    base_model_name = os.getenv("MGTD_MODEL_BASE", "/data/liyang/MODELS/gpt2-medium")

    # 扰动模型 (用于 detectGPT, fast-detectGPT)
    mask_model_name = os.getenv("MGTD_MODEL_MASK", "/data/liyang/MODELS/t5-small")

    # 双模型 (用于 Binoculars)
    binocular_observer = os.getenv(
        "MGTD_MODEL_OBSERVER", "/data/liyang/MODELS/falcon-7b"
    )
    binocular_performer = os.getenv(
        "MGTD_MODEL_PERFORMER", "/data/liyang/MODELS/falcon-7-instruct"
    )

    # 有监督参数配置
    supervised_config = {
        "need_finetune": True,
        "epochs": 1,
        "batch_size": 2,
        "lr": 5e-5,
        "pos_bit": 1,
    }


config = TestConfig()

In [ ]:
# 3. 测试无参阈值型/逻辑回归型基线 (Metric-based)
baselines_to_test = [
    ("ll", "threshold"),
    ("rank", "threshold"),
    ("entropy", "threshold"),
]

success_list = []
fail_list = []

for method_name, exp_type in baselines_to_test:
    print(f"---> Testing baseline: {method_name}")
    try:
        detector = AutoDetector.from_detector_name(
            method_name, model_name_or_path=config.base_model_name
        )
        experiment = AutoExperiment.from_experiment_name(exp_type, detector=[detector])
        experiment.load_data(mock_data)

        # Launch with configuration if necessary
        results = experiment.launch()

        for res in results:
            print(
                f"  [Result '{res.name}'] Train AUC: {res.train.auc:.4f} | Test AUC: {res.test.auc:.4f}"
            )

        success_list.append(method_name)
        print(f"✅ Baseline '{method_name}' passed!")

    except Exception as e:
        print(f"❌ Baseline '{method_name}' failed!")
        print(f"Error details: {e}")
        fail_list.append((method_name, str(e)))

In [ ]:
# 4. 测试有监督训练基线 (Supervised)
# 使用 tiny-gpt2 临时替代 Roberta 进行监督训练冒烟测试
print(f"---> Testing baseline: LM-D (Supervised)")
try:
    method_name = "LM-D"
    # 这里加载模型并指定是分类头
    detector = AutoDetector.from_detector_name(
        method_name, model_name_or_path=config.base_model_name, use_metric=False
    )
    # 给定 supervised 实验类型
    experiment = AutoExperiment.from_experiment_name("supervised", detector=[detector])
    experiment.load_data(mock_data)

    # 透传 config 给 launch
    results = experiment.launch(**config.supervised_config)

    for res in results:
        # supervised 只有 'test'/'train' 没有 name
        if res.test:
            print(f"  [Result Test] AUC: {res.test.auc:.4f}")
        if res.train:
            print(f"  [Result Train] AUC: {res.train.auc:.4f}")

    success_list.append(method_name)
    print(f"✅ Baseline '{method_name}' passed!")

except Exception as e:
    print(f"❌ Baseline '{method_name}' failed!")
    print(f"Error details: {e}")
    fail_list.append((method_name, str(e)))

In [ ]:
# 5. 测试结果汇总
print("====================================")
print(f" Summary: {len(success_list)} Passed, {len(fail_list)} Failed")
if fail_list:
    for f in fail_list:
        print(f" - {f[0]}: {f[1]}")
print("====================================")